# Stage 2 — weather + demand

This notebook is the Stage 2 demo surface. It primes two ingestion caches (NESO demand from Stage 1, Open-Meteo weather from this stage), composes per-station weather into a population-weighted national signal, joins onto hourly demand, and plots the non-linear temperature–demand relationship with a LOWESS fit.

### Why Open-Meteo, not Met Office DataHub

Open-Meteo exposes the ECMWF ERA5 / ERA5-Land / CERRA reanalyses through a free, keyless archive API (no registration, IP-throttled at 600 calls/min). The Met Office DataHub site-specific free tier only covers the last 48 hours of observations — unsuitable for a multi-year training window. Pay tiers exist but the pedagogical goal is a reproducible zero-friction notebook, so Open-Meteo wins for Stage 2.

### Data model caveat

DESIGN §4.2 describes the source as "~10 km via UKMO UKV 2 km model". The archive endpoint actually serves ERA5 / ERA5-Land / CERRA reanalyses at ~9–11 km; the UKV 2 km archive is reachable only through Open-Meteo's `historical-forecast-api` and only from 2022-03-01 — incompatible with our 2018-onwards training window. The spec is flagged for a DESIGN §4.2 correction; this notebook uses the accurate model name.

### Weighting choice

We population-weight ten city-centre stations using 2011 Census Built-Up-Area figures. This is a defensible pedagogical default — demand concentrates in population centres — but is **not** the industry standard. National Gas uses a demand-weighted Composite Weather Variable (CWV) across 13 LDZs; Thornton et al. (2016) use the unweighted Central England Temperature record. A future notebook could swap in a demand-weighted scheme and compare.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)  # cache_dir values resolve against cwd

import matplotlib.pyplot as plt  # noqa: E402
from statsmodels.nonparametric.smoothers_lowess import lowess  # noqa: E402

from bristol_ml import CachePolicy, load_config  # noqa: E402
from bristol_ml.features import weather as weather_features  # noqa: E402
from bristol_ml.ingestion import neso, weather  # noqa: E402

cfg = load_config(config_path=REPO_ROOT / "conf")
assert cfg.ingestion.neso is not None, "NESO ingestion config not resolved"
assert cfg.ingestion.weather is not None, "Weather ingestion config not resolved"

demand_path = neso.fetch(cfg.ingestion.neso, cache=CachePolicy.AUTO)
weather_path = weather.fetch(cfg.ingestion.weather, cache=CachePolicy.AUTO)
print(f"Demand cache: {demand_path}")
print(f"Weather cache: {weather_path}")

In [ ]:
demand = neso.load(Path(demand_path)).set_index("timestamp_utc").sort_index()
stations = weather.load(Path(weather_path))
weights = {s.name: s.weight for s in cfg.ingestion.weather.stations}

national_weather = weather_features.national_aggregate(stations, weights)
national_weather.head()

## Hourly join

NESO demand is half-hourly; resample to hourly means before the join. This inline resample is a deliberately thin step — Stage 3's `features/assembler.py` will re-home it into a canonical feature builder so the logic lives in one place, not every notebook.

In [ ]:
hourly_demand = demand["nd_mw"].resample("h").mean().rename("nd_mw")
joined = national_weather.join(hourly_demand, how="inner").dropna(
    subset=["temperature_2m", "nd_mw"]
)
print(f"Joined rows: {len(joined):,}")
joined.head()

## Temperature-vs-demand scatter with LOWESS fit

The GB electricity demand curve against temperature is historically a **hockey stick**, not a symmetric V: strong anti-correlation below ~17 °C, flat or slightly noisy above. The V's hot arm is emerging post-2020 as residential cooling rises, especially in London. See Thornton, Hoskins & Scaife (2016) for the pre-2014 analysis; Exeter and Tandfonline papers (2022, 2025) for the emerging hot arm.

CIBSE uses 15.5 °C as the heating-degree-day base and 22 °C as the cooling base — those are the thresholds the hockey stick's corners line up with.

In [ ]:
sample = joined
if len(sample) > 20_000:
    sample = sample.sample(20_000, random_state=0).sort_index()

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(sample["temperature_2m"], sample["nd_mw"], s=2, alpha=0.15, color="C0")

# LOWESS fit — a smooth curve, not a model
smooth = lowess(
    sample["nd_mw"].to_numpy(),
    sample["temperature_2m"].to_numpy(),
    frac=0.15,
    it=0,
    return_sorted=True,
)
ax.plot(smooth[:, 0], smooth[:, 1], color="C3", linewidth=2, label="LOWESS")

ax.axvline(15.5, color="grey", linestyle=":", alpha=0.6, label="CIBSE HDD base (15.5 °C)")
ax.axvline(22.0, color="grey", linestyle="--", alpha=0.6, label="CIBSE CDD base (22 °C)")

ax.set_xlabel("National weighted 2 m air temperature (°C)")
ax.set_ylabel("Hourly GB National Demand (MW)")
ax.set_title("Weather vs demand — hourly, population-weighted temperature")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## What the plot shows

The cold arm is textbook: a degree colder adds MW of heating-driven demand. The warm arm is flat or mildly positive — GB has historically run on gas heat and almost no residential air conditioning, so the hot side of the V is a recent, still-emerging artefact. A linear regression on this curve will badly under-predict winter peaks unless split by season or piecewise-linear above/below the HDD base. That's the motivation for the Stage 6 linear-regression + degree-days feature.

Acceptance criterion 3 — the aggregation accepts any subset. Try trimming `weights` to `{"london", "manchester", "bristol"}` and re-running the cells above: the scatter survives, the shape stays recognisable, the hot-arm wobble is noisier because three stations no longer average out regional weather.